# Padrões de codificação — Tutorial

**Programação C (COMP0512) — UFS — 2026.2**

Na aula passada o `notas.c` virou quatro arquivos: `matriz`, `estat`, `turma` e o `main`.
Hoje chega um quinto módulo, o `ranking.c`, escrito por um colega: ele classifica os alunos
pela média. O código **funciona** — e mesmo assim ninguém da equipe quer encostar nele.

Este tutorial reconstrói o projeto, roda o módulo na forma em que foi recebido, investiga o que está errado
e pede que você o traga para o padrão da disciplina.

## Objetivos

Ao final você será capaz de:

- explicar por que uma equipe adota um padrão de codificação, e o que ele fixa;
- escolher nomes pelo escopo e usar prefixo de módulo em tudo que é público;
- distinguir o comentário que repete o código do comentário que explica a decisão;
- trocar número mágico por constante com nome e variável global por parâmetro;
- usar `static` e `const` para escrever no código o que é interno e o que não muda;
- compilar com `-Wall -Wextra -Werror -pedantic` sem uma linha de aviso;
- usar `-fno-common -fsanitize=address` para transformar um estouro silencioso de memória
  em erro com arquivo e linha.

## Preparando o ambiente

Tudo acontece dentro do diretório `padroes/`. Rode esta célula uma vez.

In [ ]:
import os
os.makedirs('padroes', exist_ok=True)
%cd padroes
!gcc --version | head -1
!make --version | head -1
!clang-format --version || echo "clang-format nao encontrado. Instale com: \
apt install clang-format (Debian/Ubuntu) | brew install clang-format (macOS) | \
pip install clang-format (qualquer sistema)"


## 1. O projeto até aqui

Os quatro módulos da aula passada, sem nenhuma mudança. Repare, ao escrevê-los, em duas coisas
que já seguem o padrão de hoje: o **prefixo do módulo** nos nomes públicos (`turma_cria`,
`turma_libera`, `turma_relatorio`) e o `const` em quem só lê (`media`, `turma_relatorio`).

In [ ]:
%%writefile matriz.h
/* matriz.h --- interface: alocar e liberar matrizes de double */
#ifndef MATRIZ_H
#define MATRIZ_H

double **cria_matriz(int nl, int nc);
void libera_matriz(double **m, int nl);

#endif /* MATRIZ_H */

In [ ]:
%%writefile matriz.c
/* matriz.c --- implementacao das matrizes dinamicas */
#include <stdlib.h>
#include "matriz.h"

double **cria_matriz(int nl, int nc)
{
    double **m = malloc(nl * sizeof(double *));
    if (m == NULL)
        return NULL;
    for (int i = 0; i < nl; i++) {
        m[i] = malloc(nc * sizeof(double));
        if (m[i] == NULL) {
            for (int k = 0; k < i; k++)
                free(m[k]);
            free(m);
            return NULL;
        }
    }
    return m;
}

void libera_matriz(double **m, int nl)
{
    for (int i = 0; i < nl; i++)
        free(m[i]);
    free(m);
}

In [ ]:
%%writefile estat.h
/* estat.h --- interface: estatisticas de um vetor de notas */
#ifndef ESTAT_H
#define ESTAT_H

double media(const double *v, int n);
double desvio_padrao(const double *v, int n);

#endif /* ESTAT_H */

In [ ]:
%%writefile estat.c
/* estat.c --- implementacao das estatisticas */
#include <math.h>
#include "estat.h"

double media(const double *v, int n)
{
    double s = 0.0;
    for (int i = 0; i < n; i++)
        s += v[i];
    return n > 0 ? s / n : 0.0;
}

double desvio_padrao(const double *v, int n)
{
    double mu = media(v, n), s = 0.0;
    for (int i = 0; i < n; i++)
        s += (v[i] - mu) * (v[i] - mu);
    return n > 0 ? sqrt(s / n) : 0.0;
}

In [ ]:
%%writefile turma.h
/* turma.h --- interface: a turma e seu relatorio */
#ifndef TURMA_H
#define TURMA_H

typedef struct {
    double **notas;
    int nalunos;
    int navaliacoes;
} Turma;

Turma *turma_cria(int nalunos, int navaliacoes);
void turma_libera(Turma *t);
void turma_relatorio(const Turma *t);

#endif /* TURMA_H */

In [ ]:
%%writefile turma.c
/* turma.c --- implementacao da turma: usa matriz.h e estat.h */
#include <stdio.h>
#include <stdlib.h>
#include "turma.h"
#include "matriz.h"
#include "estat.h"

/* so turma.c enxerga esta funcao */
static void cabecalho(void)
{
    printf("%-8s %8s %8s\n", "aluno", "media", "desvio");
}

Turma *turma_cria(int nalunos, int navaliacoes)
{
    Turma *t = malloc(sizeof(Turma));
    if (t == NULL)
        return NULL;
    t->notas = cria_matriz(nalunos, navaliacoes);
    if (t->notas == NULL) {
        free(t);
        return NULL;
    }
    t->nalunos = nalunos;
    t->navaliacoes = navaliacoes;
    return t;
}

void turma_libera(Turma *t)
{
    if (t == NULL)
        return;
    libera_matriz(t->notas, t->nalunos);
    free(t);
}

void turma_relatorio(const Turma *t)
{
    cabecalho();
    for (int i = 0; i < t->nalunos; i++)
        printf("%-8d %8.2f %8.2f\n", i,
               media(t->notas[i], t->navaliacoes),
               desvio_padrao(t->notas[i], t->navaliacoes));
}

## 2. O módulo recebido

Este é o `ranking.c` na forma em que foi recebido. Não mude nada por enquanto — leia e siga em frente.

In [ ]:
%%writefile ranking.c
#include <stdio.h>
#include <stdlib.h>
#include "turma.h"
#include "estat.h"

double R[50];
int Q[50];
int N;

void troca(int a,int b){int t=Q[a];Q[a]=Q[b];Q[b]=t;}

int maior(double x,double y){ return x>y; }

void calcula(Turma *t, int flag)
{
  double tmp;
  N = t->nalunos;
  for(int i=0;i<N;i++){
  R[i]=media(t->notas[i],t->navaliacoes);
    Q[i]=i;                 /* guarda o indice i em Q[i] */
  }
	for(int i=0;i<N;i++)
		for(int j=0;j<N-1;j++)
			if(maior(R[Q[j+1]],R[Q[j]])) troca(j,j+1);
}

void mostra()
{
    printf("%-5s %-8s %8s %s\n","pos","aluno","media","situacao");
  for(int i=0;i<N;i++){
    printf("%-5d %-8d %8.2f %s\n", i+1, Q[i], R[Q[i]], R[Q[i]]>=7.0?"aprovado":"reprovado");
  }
}

E este é o `main.c` que o colega mandou junto. Repare nas duas linhas antes do `main`:
como o `ranking` **não tem header**, as assinaturas foram copiadas na mão.

In [ ]:
%%writefile main.c
/* main.c --- monta a turma e pede os relatorios */
#include <stdio.h>
#include "turma.h"

/* o ranking nao tem header: copiei as assinaturas na mao */
void calcula(Turma *t, int flag);
void mostra();

int main(void)
{
    double valores[3][4] = {
        { 8.0, 6.5, 9.0, 7.0 },
        { 5.0, 7.0, 4.0, 7.0 },
        { 9.0, 10.0, 8.5, 9.5 }
    };
    Turma *t = turma_cria(3, 4);
    if (t == NULL) {
        fprintf(stderr, "sem memoria\n");
        return 1;
    }
    for (int i = 0; i < t->nalunos; i++)
        for (int j = 0; j < t->navaliacoes; j++)
            t->notas[i][j] = valores[i][j];

    turma_relatorio(t);
    printf("\n");
    calcula(t, 0);
    mostra();

    turma_libera(t);
    return 0;
}

In [ ]:
%%writefile Makefile
CC      = gcc
CFLAGS  = -Wall -Wextra -g
LDLIBS  = -lm
OBJ     = main.o turma.o matriz.o estat.o ranking.o

notas: $(OBJ)
	$(CC) $(OBJ) -o notas $(LDLIBS)

main.o:    main.c    turma.h
turma.o:   turma.c   turma.h matriz.h estat.h
matriz.o:  matriz.c  matriz.h
estat.o:   estat.c   estat.h
ranking.o: ranking.c

.PHONY: clean
clean:
	rm -f $(OBJ) notas

### Antes de rodar, decida

Escreva suas respostas aqui (dê dois cliques nesta célula para editar):

1. O projeto compila com `-Wall -Wextra`? Com quantos avisos? → *sua resposta*
2. A saída do ranking sai certa para os três alunos do `main.c`? → *sua resposta*
3. E se a turma tiver 60 alunos em vez de 3? → *sua resposta*
4. Amanhã o `turma.c` ganha uma função chamada `troca`. O que acontece na ligação? → *sua resposta*

Só depois de responder, rode as duas células seguintes.

In [ ]:
!make 2>&1 | tail -20

In [ ]:
# guarda a saida de referencia: no fim do tutorial ela tem de ser identica
!./notas | tee saida_original.txt


## 3. Investigar

Compila, roda e acerta. É exatamente isso que torna o caso interessante: **funcionar não é
prova de estar pronto.** Vamos atrás de três coisas que a execução não mostrou.

### 3.1 Quem o módulo exporta

`nm` lista os símbolos de um arquivo-objeto. `T` é função definida aqui, `C`/`D`/`B` é variável
global, `U` é o que este arquivo *usa* mas não define. Tudo que não for `U` **sai** do arquivo e
fica visível para o ligador — e para qualquer outro módulo do projeto.

In [ ]:
!nm ranking.o | grep -E ' (T|D|B|S|C) '

Compare com a mesma consulta no `turma.o`. Lá, `cabecalho` está declarada `static` e por isso
**não aparece** — o resto do projeto não consegue chamá-la nem colidir com ela.

In [ ]:
!nm turma.o | grep -E ' (T|D|B|S|C) '

**Pergunta:** quais dos símbolos que o `ranking.o` exporta deveriam mesmo estar na interface
do módulo, e quais só vazaram?

### O que o vazamento custa

A próxima célula cria um módulo qualquer do projeto que também precisa de uma função `troca`.
Nada nele é errado — e mesmo assim a ligação quebra.

(A mensagem varia: `duplicate symbol` no macOS, `multiple definition of 'troca'` no Linux. O desfecho é o mesmo: não há executável.)


In [ ]:
%%writefile outro.c
/* outro.c --- um modulo qualquer que tambem precisa trocar coisas */
void troca(int a, int b) { (void)a; (void)b; }

In [ ]:
!gcc -c outro.c -o outro.o && gcc main.o turma.o matriz.o estat.o ranking.o outro.o -o teste -lm 2>&1 | tail -5

### 3.2 O limite escondido

`double R[50];` diz que o ranking aceita, no máximo, 50 alunos. Esse limite não está no nome de
nada, não está documentado e **não é verificado**. A célula abaixo monta uma turma de 60 alunos
com notas conhecidas — todas entre 4.0 e 10.0.

**Antes de rodar, aposte:** o programa trava? Dá mensagem de erro? Imprime resultado errado?
Anote sua aposta antes de executar.


In [ ]:
%%writefile main60.c
/* main60.c --- a mesma turma, agora com 60 alunos */
#include <stdio.h>
#include "turma.h"

void calcula(Turma *t, int flag);
void mostra();

int main(void)
{
    int n = 60;
    Turma *t = turma_cria(n, 2);
    if (t == NULL)
        return 1;
    for (int i = 0; i < n; i++) {
        t->notas[i][0] = 5.0 + (i % 6);
        t->notas[i][1] = 4.0 + (i % 7);
    }
    calcula(t, 0);
    mostra();
    turma_libera(t);
    return 0;
}

In [ ]:
!gcc -w -g main60.c turma.c matriz.c estat.c ranking.c -o notas60 -lm
!./notas60 | tail -6

Nenhuma nota entrou menor que 4.0, e ainda assim a saída não bate com isso. O laço escreveu em
`R[50]`...`R[59]`, posições que não existem.

O que está na memória logo depois de `R` **a linguagem não define**: depende do compilador, das
opções de compilação e da plataforma. Duas execuções reais deste mesmo código:

| onde | o que havia depois de `R` | o que o programa fez |
|---|---|---|
| clang no macOS (símbolos *common*) | o vetor `Q` | índices dos alunos corrompidos; últimas colocações com média `0.00` |
| gcc no Linux (`-fno-common`, padrão desde o GCC 10) | a variável `N` | a contagem virou 50: só 50 linhas impressas para 60 alunos |

Compare a sua saída com a do colega ao lado. **É normal que sejam diferentes — e é exatamente
esse o problema.** O programa não quebrou: ele mentiu, e mentiu de um jeito diferente em cada
máquina. É o desfecho normal em C quando um limite fica implícito num número solto no meio do
código.


### 3.3 Fazendo o erro aparecer

Não é preciso depender da sorte para enxergar o estouro. O *AddressSanitizer* instrumenta os
acessos à memória e aborta no ponto exato do erro, com arquivo e linha.

Uma pegadinha vale saber: ele **não** instrumenta variáveis globais que entram no `.o` como
*tentative definition* — é o caso de `double R[50];`, aquelas que o `nm` mostrou na coluna `C`.
Por isso vai junto o `-fno-common`, que faz delas objetos comuns e instrumentáveis.


In [ ]:
!gcc -w -g -fno-common -fsanitize=address \
     main60.c turma.c matriz.c estat.c ranking.c -o notas60_asan -lm
!./notas60_asan 2>&1 | head -8


Agora a mensagem diz tudo o que a execução anterior escondeu: `global-buffer-overflow`, um
`WRITE of size 8`, o arquivo e a linha. Guarde este par de flags — `-fno-common
-fsanitize=address` — porque ele economiza noites de depuração pelo resto do curso.


## 4. Tarefa 1 — formatação e nomes  *(em aula)*

Comece pela parte mecânica. O arquivo `.clang-format` abaixo é o padrão da disciplina.


In [ ]:
%%writefile .clang-format
# Padrao de formatacao da disciplina --- derivado do estilo do kernel Linux
BasedOnStyle: LLVM
IndentWidth: 4
UseTab: Never
ColumnLimit: 80
BreakBeforeBraces: Linux
AllowShortFunctionsOnASingleLine: None
AllowShortIfStatementsOnASingleLine: false
AllowShortLoopsOnASingleLine: false
SpaceBeforeParens: ControlStatements
PointerAlignment: Right
SortIncludes: false

In [ ]:
!cp ranking.c ranking_original.c
!clang-format -i ranking.c
!diff ranking_original.c ranking.c | head -40

Leia o `diff`. A ferramenta arrumou indentação, tabulações, espaços e chaves — e **não tocou**
em `R`, `Q`, `N`, nas globais nem na falta do header. Essa é a divisão de trabalho: o
`clang-format` cuida do que é mecânico; o resto exige entender o programa.

**Sua vez.** A célula abaixo já traz o arquivo como o `clang-format` o deixou. Edite os nomes
ali mesmo, sem redigitar o resto:

- `R` → `media_aluno`, `Q` → `posicao`, `N` → `quantidade`;
- `calcula` → `ranking_calcula`, `mostra` → `ranking_imprime`;
- apague o parâmetro `flag` e a variável `tmp`, que ninguém usa;
- troque o `7.0` solto pela constante com nome que já está declarada no topo.

Mantenha as globais e a ausência de header por enquanto — isso é a Tarefa 2. **Não mexa nos
`printf`:** ao final, `make && ./notas` tem de imprimir exatamente a mesma coisa de antes, e a
célula de verificação vai comparar com o `saida_original.txt` linha por linha.


In [ ]:
%%writefile ranking.c
#include <stdio.h>
#include <stdlib.h>
#include "turma.h"
#include "estat.h"

/* Media minima para aprovacao --- ja com nome, em vez do 7.0 solto */
#define RANKING_NOTA_CORTE 7.0

/* TODO: renomeie as tres globais (elas somem na Tarefa 2) */
double R[50];
int Q[50];
int N;

void troca(int a, int b)
{
    int t = Q[a];
    Q[a] = Q[b];
    Q[b] = t;
}

int maior(double x, double y)
{
    return x > y;
}

/* TODO: renomeie para ranking_calcula e apague o parametro flag */
void calcula(Turma *t, int flag)
{
    double tmp;  /* TODO: ninguem usa --- apague */
    N = t->nalunos;
    for (int i = 0; i < N; i++) {
        R[i] = media(t->notas[i], t->navaliacoes);
        Q[i] = i;  /* TODO: este comentario so repetia o codigo */
    }
    for (int i = 0; i < N; i++)
        for (int j = 0; j < N - 1; j++)
            if (maior(R[Q[j + 1]], R[Q[j]]))
                troca(j, j + 1);
}

/* TODO: renomeie para ranking_imprime */
void mostra()
{
    printf("%-5s %-8s %8s %s\n", "pos", "aluno", "media", "situacao");
    for (int i = 0; i < N; i++) {
        printf("%-5d %-8d %8.2f %s\n", i + 1, Q[i], R[Q[i]],
               R[Q[i]] >= 7.0 ? "aprovado" : "reprovado");  /* TODO: use a constante */
    }
}


In [ ]:
# Ajuste as assinaturas copiadas no main.c para os nomes novos e recompile.
# A saida tem de ser identica a da secao 2.
!make 2>&1 | tail -10
!./notas

## 5. Tarefa 2 — o header e o fim das globais  *(começa em aula, termina em casa)*

Agora a parte que a ferramenta não faz. O estado sai das globais e passa a viver numa `struct`
que o próprio módulo aloca e devolve; tudo que não estiver no header vira `static`.

Complete o `ranking.h` abaixo. Ele é o **contrato**: quem for usar o módulo lê só este arquivo.

In [ ]:
%%writefile ranking.h
/* ranking.h --- interface: classificacao dos alunos de uma turma pela media */
#ifndef RANKING_H
#define RANKING_H

#include "turma.h"

/* Media minima para aprovacao; faz parte do contrato do modulo. */
#define RANKING_NOTA_CORTE 7.0

typedef struct {
    int    *posicao;   /* posicao[0] = indice do aluno de maior media */
    double *media;     /* media[i] = media do aluno de indice i */
    int     quantidade;
} Ranking;

/* TODO: documente cada funcao --- o que faz, o que devolve no erro,
   e de quem e a responsabilidade de liberar a memoria. */
Ranking *ranking_cria(const Turma *t);
void     ranking_libera(Ranking *r);
int      ranking_aprovado(const Ranking *r, int colocacao);
void     ranking_imprime(const Ranking *r);

#endif /* RANKING_H */

E agora a implementação. Os esqueletos indicam o que cada função deve fazer; note que
`ordena` é `static` — ela não está no header, logo não sai do arquivo.

In [ ]:
%%writefile ranking.c
/* ranking.c --- implementacao da classificacao por media */
#include <stdio.h>
#include <stdlib.h>
#include "ranking.h"
#include "estat.h"

/* Ordena as posicoes por media decrescente.
   TODO: escreva aqui *por que* este algoritmo, e nao outro. */
static void ordena(Ranking *r)
{
    /* TODO: ordene r->posicao usando r->media como chave */
}

Ranking *ranking_cria(const Turma *t)
{
    /* TODO: aloque o Ranking e os dois vetores com o tamanho da turma
       (nada de 50), preencha as medias, chame ordena e devolva.
       Devolva NULL se faltar memoria --- sem vazar o que ja alocou. */
    return NULL;
}

void ranking_libera(Ranking *r)
{
    /* TODO: aceite NULL e libere os dois vetores antes da struct */
}

int ranking_aprovado(const Ranking *r, int colocacao)
{
    /* TODO: use RANKING_NOTA_CORTE */
    return 0;
}

void ranking_imprime(const Ranking *r)
{
    /* TODO: uma linha por aluno, na ordem do ranking. Use os mesmos
       printf da Tarefa 1: a saida tem de ficar identica a original. */
}

O `main.c` abaixo já está no padrão: **nenhuma assinatura copiada na mão**, só o `#include`.
Ele também trata o erro de alocação e libera o que criou.

In [ ]:
%%writefile main.c
/* main.c --- monta a turma e pede os relatorios */
#include <stdio.h>
#include "turma.h"
#include "ranking.h"

int main(void)
{
    double valores[3][4] = {
        { 8.0, 6.5, 9.0, 7.0 },
        { 5.0, 7.0, 4.0, 7.0 },
        { 9.0, 10.0, 8.5, 9.5 }
    };
    Turma *t = turma_cria(3, 4);
    if (t == NULL) {
        fprintf(stderr, "sem memoria\n");
        return 1;
    }
    for (int i = 0; i < t->nalunos; i++)
        for (int j = 0; j < t->navaliacoes; j++)
            t->notas[i][j] = valores[i][j];

    turma_relatorio(t);
    printf("\n");

    Ranking *r = ranking_cria(t);
    if (r == NULL) {
        fprintf(stderr, "sem memoria para o ranking\n");
        turma_libera(t);
        return 1;
    }
    ranking_imprime(r);

    ranking_libera(r);
    turma_libera(t);
    return 0;
}

## 6. Tarefa 3 — o padrão que a máquina cobra  *(em casa)*

Duas mudanças no `Makefile`: as flags que transformam aviso em erro, e as dependências do
`ranking.o`, que hoje mentem (ele inclui `ranking.h`, `turma.h` e `estat.h`, e o `Makefile` não
sabe disso).

In [ ]:
%%writefile Makefile
CC      = gcc
# TODO: acrescente -std=c17 -Werror -pedantic
CFLAGS  = -Wall -Wextra -g
LDLIBS  = -lm
OBJ     = main.o turma.o matriz.o estat.o ranking.o

notas: $(OBJ)
	$(CC) $(OBJ) -o notas $(LDLIBS)

main.o:    main.c    turma.h
turma.o:   turma.c   turma.h matriz.h estat.h
matriz.o:  matriz.c  matriz.h
estat.o:   estat.c   estat.h
# TODO: corrija as dependencias abaixo
ranking.o: ranking.c

.PHONY: clean formato
clean:
	rm -f $(OBJ) notas

formato:
	clang-format -i *.c *.h

### Verificação

A célula abaixo é o seu critério de pronto. Ela recompila do zero e confere três coisas: que a
compilação não emitiu **nenhuma** linha de aviso, que o `ranking.o` não exporta mais nada além
da interface, e que a saída é **byte por byte** a mesma que você guardou no começo do tutorial.


In [ ]:
import subprocess

subprocess.run(['make', 'clean'], capture_output=True)
build = subprocess.run(['make'], capture_output=True, text=True)
saida = build.stdout + build.stderr

# 'warning:'/'error:' com dois-pontos: a flag -Werror da linha de comando nao conta
if 'warning:' in saida or 'error:' in saida:
    print('AINDA HA AVISOS:\n')
    print(saida)
else:
    print('Compilou limpo.\n')

    simbolos = subprocess.run('nm ranking.o | grep -E " (T|D|B|S|C) "',
                              shell=True, capture_output=True, text=True).stdout
    print('Simbolos exportados por ranking.o:')
    print(simbolos)

    atual = subprocess.run(['./notas'], capture_output=True, text=True).stdout
    original = open('saida_original.txt').read()
    print(atual)
    if atual != original:
        import difflib
        print('A SAIDA MUDOU:')
        print(''.join(difflib.unified_diff(original.splitlines(True),
                                           atual.splitlines(True),
                                           'saida_original.txt', './notas')))
    else:
        print('Saida identica, byte por byte, a do inicio do tutorial.')


## Desafio  *(em casa)*

Hoje o ranking sempre ordena por média. Faça o critério de ordenação virar parte da interface,
**sem tocar em `turma.c` nem em `estat.c`**:

- acrescente ao `ranking.h` um tipo para o critério — por média, por maior nota, por menor
  desvio;
- `ranking_cria` passa a receber o critério; cada função de comparação fica `static` dentro do
  `ranking.c`;
- documente no header o que cada critério faz **e o que acontece em caso de empate**;
- compile com `-Werror` e mostre, com `nm ranking.o`, que nenhum símbolo novo vazou para fora
  do módulo.

Pergunta para responder junto com o código: por que o `turma.c` não precisou ser recompilado?

In [ ]:
# Desafio --- comece pelo header: o contrato vem antes da implementacao.
# %%writefile ranking.h


## Referências

- Kernighan, B. W.; Pike, R. *The Practice of Programming*. Addison-Wesley, 1999 — cap. 1,
  "Style": nomes, consistência, números mágicos e comentários.
- McConnell, S. *Code Complete*. 2. ed. Microsoft Press, 2004 — cap. 11 (nomes), cap. 31
  (layout) e cap. 32 (código autodocumentado).
- Kernighan, B. W.; Ritchie, D. M. *The C Programming Language*. 2. ed. Prentice Hall, 1988 —
  §4.6 sobre `static`.
- *Linux Kernel Coding Style* — https://www.kernel.org/doc/html/latest/process/coding-style.html
- *GNU Coding Standards* — https://www.gnu.org/prep/standards/
- Backes, A. *Linguagem C: completa e descomplicada*. Elsevier, 2013.

A lista completa está em `../referencias.bib`.